In [1]:
!pip install -q transformers datasets evaluate accelerate


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import ViTImageProcessor, ViTForImageClassification, TrainingArguments, Trainer
import evaluate

/home/lyj/venv/myvenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


문제 1 (단답형 주관식)

기존 CNN(합성곱 신경망)이 이미지를 2차원 그리드(픽셀) 그대로 처리하는 것과 달리, Vision Transformer(ViT)는 NLP의 트랜스포머 구조를 사용하기 위해 이미지를 1차원 시퀀스(Sequence) 데이터로 변환해야 합니다.

이때 이미지를 고정된 크기의 작은 조각으로 자르는 과정을 무엇이라 부르며, 이 조각들을 신경망에 입력하기 위해 벡터로 변환(Flatten & Linear Projection)하는 과정을 통칭하여 무엇(Embedding)이라고 합니까?


1번 답: 작은 조각의 명칭은 패치이며, 변환 과정의 명칭은 패치 임베딩이다. ViT는 이미지를 16x16 픽셀 크기의 패치로 잘라서 각 패치를 Flatten 후 Linear Projection을 통해 벡터로 변환한다.

문제 2 (단답형 주관식)

ViT 모델의 입력 시퀀스 맨 앞에 추가되는 학습 가능한 특수 토큰(Learnable Token)이 있습니다.

트랜스포머 인코더를 통과한 후, 이 토큰의 출력 벡터가 이미지 전체의 정보를 요약하고 있다고 가정하여 최종 분류(Classification)에 사용됩니다.

이 토큰의 이름은 무엇입니까? (힌트: BERT 모델에서도 유사한 토큰을 사용합니다.)

2번 답: CLS 토큰, BERT에서도 동일하게 쓰이는 토큰으로, 시퀀스 맨 앞에 붙여 트랜스포머를 통과시킨 후 그 출력 벡터로 최종 분류를 수행.

문제 3 (실습 문제 - 코드 빈칸 채우기)

Hugging Face의 transformers 라이브러리를 사용하여, 구글의 사전 학습된 ViT 모델(google/vit-base-patch16-224-in21k)에 맞는 이미지 전처리기(Processor)를 로드하는 코드입니다.

빈칸 (# TODO: ...)을 채우시오.

In [3]:
from transformers import ViTImageProcessor

model_name = 'google/vit-base-patch16-224-in21k'

# TODO: 1. 위 model_name에 해당하는 사전 학습된 'processor' 로드
# (힌트: from_pretrained 메소드 사용)
processor = ViTImageProcessor.from_pretrained(model_name)

# --- 결과 확인 (수정 불필요) ---
print(processor)
# (출력 예시: size={'height': 224, 'width': 224} 등 전처리 설정 정보가 출력됨)

ViTImageProcessor {
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "ViTImageProcessor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "resample": 2,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "height": 224,
    "width": 224
  }
}



문제 4 (실습 문제 - 코드 빈칸 채우기)

로드한 processor를 사용하여 데이터셋(example_batch)의 이미지를 전처리하는 transform 함수를 정의하는 코드입니다.

processor는 이미지를 모델 입력에 맞는 텐서(pixel_values)로 변환해줍니다.

빈칸을 채워 함수를 완성하시오.

In [4]:
def transform(example_batch):
    # example_batch['image']: 이미지 객체들의 리스트

    # TODO: 1. processor를 사용하여 이미지를 전처리하고 텐서로 변환(return_tensors='pt')
    # (힌트: images 인자에 이미지 리스트를 전달)
    inputs = processor(
        images = example_batch['image'],
        return_tensors = 'pt'
    )

    # TODO: 2. 변환된 텐서('pixel_values')를 example_batch에 저장하여 반환
    example_batch['pixel_values'] = inputs['pixel_values']
    return example_batch

# --- 가상 데이터 테스트 (수정 불필요) ---
# (더미 이미지 데이터 생성 및 테스트)
from PIL import Image
import numpy as np
dummy_data = {'image': [Image.fromarray(np.uint8(np.random.rand(100, 100, 3) * 255)) for _ in range(2)]}
processed_data = transform(dummy_data)

print(f"전처리된 텐서 크기: {processed_data['pixel_values'].shape}")
# (예상 출력: [2, 3, 224, 224])

전처리된 텐서 크기: torch.Size([2, 3, 224, 224])


문제 5 (실습 문제 - 코드 빈칸 채우기)

이미지 분류를 위한 ViTForImageClassification 모델을 로드하는 코드입니다.
이때, 데이터셋의 레이블 수(num_labels)와 레이블 매핑 정보(id2label, label2id)를 함께 설정해야 올바른 분류 헤드(Classification Head)가 생성됩니다.

3개의 빈칸 (# TODO: ...)을 채우시오.

In [5]:
from transformers import ViTForImageClassification

# --- 가상의 레이블 정보 (수정 불필요) ---
labels = ['cat', 'dog', 'bird'] # 3개 클래스라고 가정
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}
# ---------------------------------------

# TODO: 1. 사전 학습된 ViT 분류 모델 로드
model = ViTForImageClassification.from_pretrained(
    model_name,

    # TODO: 2. 클래스 개수 설정
    num_labels = 3,

    # TODO: 3. ID <-> Label 매핑 정보 설정
    id2label = id2label,
    label2id = label2id
)

# --- 결과 확인 (수정 불필요) ---
print(f"모델의 분류기 출력 크기: {model.classifier.out_features}")
# (예상 출력: 3)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 31563.69it/s]
[transformers] ViTForImageClassification LOAD REPORT from: google/vit-base-patch16-224-in21k
Key                 | Status     | 
--------------------+------------+-
pooler.dense.weight | UNEXPECTED | 
pooler.dense.bias   | UNEXPECTED | 
classifier.weight   | MISSING    | 
classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


모델의 분류기 출력 크기: 3


문제 6 (실습 문제 - 코드 작성)

Hugging Face의 Trainer를 사용하여 모델을 학습시키기 위한 설정(TrainingArguments)과 Trainer 객체를 생성하는 코드입니다.

2개의 빈칸 (# TODO: ...)을 채우시오.

[요구사항]
- TrainingArguments: 학습 결과를 저장할 디렉토리(output_dir)를 './results'로 설정하고, 에포크 당 평가(evaluation_strategy)를 'epoch'으로 설정합니다.
- Trainer: 위에서 준비한 model, args, train_dataset, eval_dataset 등을 인자로 전달하여 초기화합니다. (가상 데이터셋 변수 사용)


In [6]:
from transformers import TrainingArguments, Trainer
from transformers import DefaultDataCollator

# --- 가상 데이터셋 및 Collator (수정 불필요) ---
# (실제 데이터셋 대신 None으로 자리만 표시함)
train_ds = None
test_ds = None
data_collator = DefaultDataCollator()
# ----------------------------------------------

# TODO: 1. 학습 인자(TrainingArguments) 설정
training_args = TrainingArguments(
    output_dir = './results',
    evaluation_strategy = 'epoch',
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=1,
    remove_unused_columns=False
)

# TODO: 2. Trainer 객체 생성
trainer = Trainer(
    model = model,
    args = training_args,          # 위에서 정의한 인자
    data_collator = data_collator,
    train_dataset = train_ds,
    eval_dataset = test_ds,
    tokenizer = processor # (ViT에서는 tokenizer 인자에 processor를 전달)
)

# --- 결과 확인 (수정 불필요) ---
print(f"Trainer 설정 완료: {type(trainer)}")

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [3]:
# EOS